# Rumi – Sindh Users Data Analysis

**Purpose**: Detailed analysis of teachers using Rumi from Sindh province (Pakistan).

**Identification strategy**:
- Phone area codes: landline prefixes for Karachi (9221), Hyderabad (9222), Sukkur (9251), Larkana (9243), etc.
- School-name keyword matching: Karachi, Hyderabad, Sindh, Clifton, Defence, Gulshan, etc.

> ⚠️ Mobile numbers don't carry province info — school-name matching is the stronger signal for WhatsApp users.

In [ ]:
import os, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.1f}'.format)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)

DB_URL = (
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
    f"?sslmode={os.getenv('DB_SSL', 'require')}"
)
engine = create_engine(DB_URL, connect_args={'connect_timeout': 30})

def q(sql):
    with engine.connect() as c:
        return pd.read_sql(text(sql), c)

print('Engine ready.')

## 1. Identify Sindh Users

In [ ]:
SINDH_KEYWORDS = [
    'karachi','hyderabad','sukkur','larkana','nawabshah',
    'mirpurkhas','khairpur','jacobabad','shikarpur','ghotki',
    'dadu','thatta','badin','sanghar','tharparkar','mithi',
    'umerkot','matiari','jamshoro','kamber','shahdadkot',
    'kotri','clifton','defence','gulshan','nazimabad',
    'saddar','malir','landhi','korangi','orangi','baldia',
    'sindh','sind',
]

SINDH_PHONE_PREFIXES = [
    '9221','9222','9223','9235','9241','9242','9243',
    '9244','9251','9261','9291','9292','9297','9298','9233',
]

school_sql = ' OR '.join(f"LOWER(school_name) LIKE '%{kw}%'" for kw in SINDH_KEYWORDS)
phone_sql  = ' OR '.join(f"phone_number LIKE '{p}%'" for p in SINDH_PHONE_PREFIXES)

users = q(f"""
    SELECT
        id, phone_number, first_name, last_name, school_name,
        grades_taught, subjects_taught, preferred_language,
        registration_completed, registration_state,
        portal_activated, created_at, updated_at
    FROM users
    WHERE COALESCE(is_test_user, false) = false
      AND LEFT(phone_number, 2) = '92'
      AND (({phone_sql}) OR ({school_sql}))
    ORDER BY created_at DESC
""")

print(f'Sindh users found: {len(users):,}')
users.head(10)

## 2. High-Level Summary

In [ ]:
IDS = ','.join(f"'{i}'" for i in users['id'].tolist())

summary = q(f"""
    SELECT
        COUNT(*)                                                          AS total_users,
        COUNT(*) FILTER (WHERE registration_completed)                    AS registered,
        COUNT(*) FILTER (WHERE NOT registration_completed)                AS unregistered,
        ROUND(100.0 * COUNT(*) FILTER (WHERE registration_completed)
              / NULLIF(COUNT(*),0),1)                                     AS reg_rate_pct,
        COUNT(*) FILTER (WHERE preferred_language = 'ur')                 AS urdu,
        COUNT(*) FILTER (WHERE preferred_language = 'en')                 AS english,
        COUNT(*) FILTER (WHERE preferred_language = 'ar')                 AS arabic,
        MIN(created_at)::date                                             AS first_join,
        MAX(created_at)::date                                             AS latest_join
    FROM users
    WHERE id = ANY(ARRAY[{IDS}]::uuid[])
""")
summary.T

## 3. Monthly User Growth

In [ ]:
growth = q(f"""
    SELECT
        TO_CHAR(DATE_TRUNC('month', created_at),'YYYY-MM') AS month,
        COUNT(*)                                            AS new_users,
        COUNT(*) FILTER (WHERE registration_completed)      AS registered
    FROM users
    WHERE id = ANY(ARRAY[{IDS}]::uuid[])
    GROUP BY DATE_TRUNC('month', created_at)
    ORDER BY DATE_TRUNC('month', created_at)
""")

fig, ax = plt.subplots()
ax.bar(growth['month'], growth['new_users'], label='All users', alpha=0.8)
ax.bar(growth['month'], growth['registered'], label='Registered', alpha=0.8)
ax.set_title('Sindh – Monthly New Users')
ax.set_xlabel('Month'); ax.set_ylabel('Users')
plt.xticks(rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('sindh_monthly_growth.png', dpi=150)
plt.show()
growth

## 4. Messaging Engagement

In [ ]:
engagement = q(f"""
    SELECT
        COUNT(*)                                           AS total_messages,
        COUNT(DISTINCT user_id)                            AS active_users,
        COUNT(*) FILTER (WHERE message_type='text')        AS text_msgs,
        COUNT(*) FILTER (WHERE message_type='voice')       AS voice_msgs,
        COUNT(*) FILTER (WHERE message_type='image')       AS image_msgs,
        ROUND(100.0 * COUNT(*) FILTER (WHERE message_type='voice')
              / NULLIF(COUNT(*),0), 1)                     AS voice_pct
    FROM conversations
    WHERE user_id = ANY(ARRAY[{IDS}]::uuid[]) AND role='user'
""")

# Message type pie chart
e = engagement.iloc[0]
labels = ['Text', 'Voice', 'Image']
vals   = [e['text_msgs'], e['voice_msgs'], e['image_msgs']]
fig, ax = plt.subplots(figsize=(6,6))
ax.pie(vals, labels=labels, autopct='%1.1f%%', startangle=140)
ax.set_title('Sindh – Message Type Distribution')
plt.tight_layout()
plt.savefig('sindh_message_types.png', dpi=150)
plt.show()
engagement.T

## 5. Feature Usage

In [ ]:
features = q(f"""
    SELECT 'Lesson Plans (completed)' AS feature, COUNT(*) AS count
    FROM lesson_plan_requests WHERE user_id=ANY(ARRAY[{IDS}]::uuid[]) AND status='completed'
    UNION ALL
    SELECT 'Lesson Plans (all)',      COUNT(*) FROM lesson_plan_requests WHERE user_id=ANY(ARRAY[{IDS}]::uuid[])
    UNION ALL
    SELECT 'Coaching (completed)',    COUNT(*) FROM coaching_sessions WHERE user_id=ANY(ARRAY[{IDS}]::uuid[]) AND status='completed'
    UNION ALL
    SELECT 'Coaching (all)',          COUNT(*) FROM coaching_sessions WHERE user_id=ANY(ARRAY[{IDS}]::uuid[])
    UNION ALL
    SELECT 'Reading Assessments (completed)', COUNT(*) FROM reading_assessments WHERE user_id=ANY(ARRAY[{IDS}]::uuid[]) AND status='completed'
    UNION ALL
    SELECT 'Video Requests (completed)', COUNT(*) FROM video_requests WHERE user_id=ANY(ARRAY[{IDS}]::uuid[]) AND status='completed'
    UNION ALL
    SELECT 'Image Analysis (completed)', COUNT(*) FROM image_analysis_requests WHERE user_id=ANY(ARRAY[{IDS}]::uuid[]) AND status='completed'
    ORDER BY count DESC
""")

fig, ax = plt.subplots(figsize=(10,5))
ax.barh(features['feature'], features['count'], color=sns.color_palette('Set2', len(features)))
ax.set_xlabel('Count')
ax.set_title('Sindh – Feature Usage')
plt.tight_layout()
plt.savefig('sindh_feature_usage.png', dpi=150)
plt.show()
features

## 6. Language Preferences

In [ ]:
lang = users['preferred_language'].value_counts().reset_index()
lang.columns = ['language', 'count']

fig, ax = plt.subplots(figsize=(7,5))
ax.bar(lang['language'].fillna('unknown'), lang['count'], color=sns.color_palette('Set2', len(lang)))
ax.set_title('Sindh – Language Preferences')
ax.set_xlabel('Language'); ax.set_ylabel('Users')
plt.tight_layout()
plt.savefig('sindh_language_pref.png', dpi=150)
plt.show()
lang

## 7. Top Schools

In [ ]:
schools = q(f"""
    SELECT school_name, COUNT(*) AS teachers,
           COUNT(*) FILTER (WHERE registration_completed) AS registered
    FROM users
    WHERE id = ANY(ARRAY[{IDS}]::uuid[])
      AND school_name IS NOT NULL AND school_name <> ''
    GROUP BY school_name
    ORDER BY teachers DESC
    LIMIT 25
""")
schools

## 8. Grades Taught Distribution

In [ ]:
grades = q(f"""
    SELECT grades_taught, COUNT(*) AS users
    FROM users
    WHERE id = ANY(ARRAY[{IDS}]::uuid[])
      AND grades_taught IS NOT NULL AND grades_taught <> ''
    GROUP BY grades_taught
    ORDER BY users DESC
    LIMIT 20
""")
grades

## 9. Reading Assessment Benchmarks

In [ ]:
reading = q(f"""
    SELECT language, grade_level,
           COUNT(*)                                                          AS assessments,
           ROUND(AVG(wcpm),1)                                                AS avg_wcpm,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY wcpm),1)       AS median_wcpm,
           COUNT(*) FILTER (WHERE benchmark_status='above')                  AS above,
           COUNT(*) FILTER (WHERE benchmark_status='at')                     AS at_level,
           COUNT(*) FILTER (WHERE benchmark_status='below')                  AS below
    FROM reading_assessments
    WHERE user_id = ANY(ARRAY[{IDS}]::uuid[])
      AND status='completed' AND wcpm IS NOT NULL
    GROUP BY language, grade_level
    ORDER BY language, grade_level
""")
reading

## 10. Power Users (Top 20)

In [ ]:
power = q(f"""
    SELECT
        u.phone_number, u.first_name, u.school_name, u.grades_taught,
        u.preferred_language, u.created_at::date AS joined,
        (SELECT COUNT(*) FROM lesson_plan_requests l WHERE l.user_id=u.id AND l.status='completed') AS lesson_plans,
        (SELECT COUNT(*) FROM coaching_sessions    c WHERE c.user_id=u.id AND c.status='completed') AS coaching,
        (SELECT COUNT(*) FROM reading_assessments  r WHERE r.user_id=u.id AND r.status='completed') AS reading,
        (SELECT COUNT(*) FROM image_analysis_requests i WHERE i.user_id=u.id AND i.status='completed') AS images,
        (SELECT COUNT(*) FROM conversations        v WHERE v.user_id=u.id AND v.role='user') AS messages
    FROM users u
    WHERE u.id = ANY(ARRAY[{IDS}]::uuid[]) AND u.registration_completed = true
    ORDER BY
        (SELECT COUNT(*) FROM lesson_plan_requests l WHERE l.user_id=u.id) +
        (SELECT COUNT(*) FROM coaching_sessions    c WHERE c.user_id=u.id) +
        (SELECT COUNT(*) FROM reading_assessments  r WHERE r.user_id=u.id) DESC
    LIMIT 20
""")
power

## 11. Export Results to Excel

In [ ]:
with pd.ExcelWriter('sindh_report.xlsx', engine='openpyxl') as writer:
    users.drop(columns=['id'], errors='ignore').to_excel(writer, sheet_name='Sindh Users', index=False)
    summary.T.to_excel(writer, sheet_name='Summary')
    features.to_excel(writer, sheet_name='Feature Usage', index=False)
    engagement.T.to_excel(writer, sheet_name='Engagement')
    growth.to_excel(writer, sheet_name='Monthly Growth', index=False)
    schools.to_excel(writer, sheet_name='Top Schools', index=False)
    grades.to_excel(writer, sheet_name='Grades', index=False)
    power.to_excel(writer, sheet_name='Power Users', index=False)
    if not reading.empty:
        reading.to_excel(writer, sheet_name='Reading Benchmarks', index=False)

print('Exported → sindh_report.xlsx')